In [ ]:
%pip install numpy matplotlib opencv-contrib-python

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

OUTPUT_FOLDER = Path("output")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)


In [ ]:
# The exploratory cells use the first supplied pair (ski scene).
image1_path = Path("images/leftImage.png")
image2_path = Path("images/rightImage.png")
image1 = cv2.imread(str(image1_path))
image2 = cv2.imread(str(image2_path))
if image1 is None or image2 is None:
    raise FileNotFoundError(f"Could not load {image1_path} and {image2_path}.")
print("Image 1 shape:", image1.shape)
print("Image 2 shape:", image2.shape)


### Display images in RGB

OpenCV reads the files as BGR, so I convert them to RGB for plotting.

In [ ]:

# Convert BGR to RGB
image1_rgb = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)
image2_rgb = cv2.cvtColor(image2, cv2.COLOR_BGR2RGB)

# Display both images
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(image1_rgb)
axes[0].set_title("Image 1")
axes[0].axis("off")

axes[1].imshow(image2_rgb)
axes[1].set_title("Image 2")
axes[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

print("Image 1 dimensions:", image1.shape[:2])
print("Image 2 dimensions:", image2.shape[:2])

print("Image 1 height:", image1.shape[0])
print("Image 1 width:", image1.shape[1])

print("Image 2 height:", image2.shape[0])
print("Image 2 width:", image2.shape[1])


## Harris corners

In [ ]:
# Convert images from BGR to grayscale
gray1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)

# Convert pixel values to float32 and normalize to [0, 1]
gray1 = gray1.astype(np.float32) / 255.0
gray2 = gray2.astype(np.float32) / 255.0

# Display grayscale images
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(gray1, cmap="gray")
axes[0].set_title("Image 1 - Grayscale")
axes[0].axis("off")

axes[1].imshow(gray2, cmap="gray")
axes[1].set_title("Image 2 - Grayscale")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Calculate horizontal and vertical image gradients
Ix1 = cv2.Sobel(
    gray1, cv2.CV_32F, 1, 0, ksize=3
)

Iy1 = cv2.Sobel(
    gray1, cv2.CV_32F, 0, 1, ksize=3
)

Ix2 = cv2.Sobel(
    gray2, cv2.CV_32F, 1, 0, ksize=3
)

Iy2 = cv2.Sobel(
    gray2, cv2.CV_32F, 0, 1, ksize=3
)

### Image gradients

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Image 1
axes[0, 0].imshow(gray1, cmap="gray")
axes[0, 0].set_title("Original grayscale")

axes[0, 1].imshow(Ix1, cmap="gray")
axes[0, 1].set_title("Horizontal gradient (Ix)")

axes[0, 2].imshow(Iy1, cmap="gray")
axes[0, 2].set_title("Vertical gradient (Iy)")

# Image 2
axes[1, 0].imshow(gray2, cmap="gray")
axes[1, 0].set_title("Original grayscale")

axes[1, 1].imshow(Ix2, cmap="gray")
axes[1, 1].set_title("Horizontal gradient (Ix)")

axes[1, 2].imshow(Iy2, cmap="gray")
axes[1, 2].set_title("Vertical gradient (Iy)")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Calculate products of image gradients
Ix1_squared = Ix1 ** 2
Iy1_squared = Iy1 ** 2
Ixy1 = Ix1 * Iy1

Ix2_squared = Ix2 ** 2
Iy2_squared = Iy2 ** 2
Ixy2 = Ix2 * Iy2

window_size = 5
sigma = 1

# Image 1
Sxx1 = cv2.GaussianBlur(
    Ix1_squared, (window_size, window_size), sigma
)
Syy1 = cv2.GaussianBlur(
    Iy1_squared, (window_size, window_size), sigma
)
Sxy1 = cv2.GaussianBlur(
    Ixy1, (window_size, window_size), sigma
)

# Image 2
Sxx2 = cv2.GaussianBlur(
    Ix2_squared, (window_size, window_size), sigma
)
Syy2 = cv2.GaussianBlur(
    Iy2_squared, (window_size, window_size), sigma
)
Sxy2 = cv2.GaussianBlur(
    Ixy2, (window_size, window_size), sigma
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Image 1
axes[0, 0].imshow(Sxx1, cmap="hot")
axes[0, 0].set_title("Image 1 - Sxx")

axes[0, 1].imshow(Syy1, cmap="hot")
axes[0, 1].set_title("Image 1 - Syy")

axes[0, 2].imshow(Sxy1, cmap="coolwarm")
axes[0, 2].set_title("Image 1 - Sxy")

# Image 2
axes[1, 0].imshow(Sxx2, cmap="hot")
axes[1, 0].set_title("Image 2 - Sxx")

axes[1, 1].imshow(Syy2, cmap="hot")
axes[1, 1].set_title("Image 2 - Syy")

axes[1, 2].imshow(Sxy2, cmap="coolwarm")
axes[1, 2].set_title("Image 2 - Sxy")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Harris detector parameter
alpha = 0.04

# Calculate Harris response for Image 1
det1 = (Sxx1 * Syy1) - (Sxy1 ** 2)
trace1 = Sxx1 + Syy1

R1 = det1 - alpha * (trace1 ** 2)

# Calculate Harris response for Image 2
det2 = (Sxx2 * Syy2) - (Sxy2 ** 2)
trace2 = Sxx2 + Syy2

R2 = det2 - alpha * (trace2 ** 2)

## Harris response

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(R1, cmap="coolwarm")
axes[0].set_title("Harris Response - Image 1")

axes[1].imshow(R2, cmap="coolwarm")
axes[1].set_title("Harris Response - Image 2")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("Image 1:")
print("Minimum response:", R1.min())
print("Maximum response:", R1.max())

print("\nImage 2:")
print("Minimum response:", R2.min())
print("Maximum response:", R2.max())

### Threshold and non-maximum suppression

In [ ]:
def detect_keypoints(R, threshold_ratio=0.01, nms_size=5):

    # Step 1: Calculate the threshold
    threshold = threshold_ratio * R.max()

    # Step 2: Find local maximum values
    kernel = np.ones((nms_size, nms_size), np.uint8)
    local_max = cv2.dilate(R, kernel)

    # Step 3: Keep pixels that are both:
    # - Above the threshold
    # - Equal to the local maximum
    keypoint_mask = (R > threshold) & (R == local_max)

    # Step 4: Extract their (y, x) coordinates
    keypoints = np.argwhere(keypoint_mask)

    # Step 5: Sort keypoints by Harris response
    responses = R[keypoints[:, 0], keypoints[:, 1]]
    order = np.argsort(responses)[::-1]

    keypoints = keypoints[order]

    return keypoints

In [ ]:
# Detect keypoints in both images
keypoints1 = detect_keypoints(R1)
keypoints2 = detect_keypoints(R2)

print("Image 1 keypoints:", len(keypoints1))
print("Image 2 keypoints:", len(keypoints2))

### Detected keypoints

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Convert original images to RGB for plotting
image1_rgb = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)
image2_rgb = cv2.cvtColor(image2, cv2.COLOR_BGR2RGB)

# Image 1
axes[0].imshow(image1_rgb)
axes[0].scatter(
    keypoints1[:, 1],
    keypoints1[:, 0],
    s=8,
    c="red"
)
axes[0].set_title(f"Image 1 - {len(keypoints1)} keypoints")
axes[0].axis("off")

# Image 2
axes[1].imshow(image2_rgb)
axes[1].scatter(
    keypoints2[:, 1],
    keypoints2[:, 0],
    s=8,
    c="red"
)
axes[1].set_title(f"Image 2 - {len(keypoints2)} keypoints")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### Extract image patches

In [ ]:
def extract_patches(image, keypoints, patch_size=11):

    patches = []
    valid_keypoints = []

    radius = patch_size // 2

    height, width = image.shape

    for y, x in keypoints:

        # Skip keypoints too close to image boundaries
        if (y - radius < 0 or
            y + radius >= height or
            x - radius < 0 or
            x + radius >= width):
            continue

        # Extract a square patch around the keypoint
        patch = image[
            y-radius:y+radius+1,
            x-radius:x+radius+1
        ]

        patches.append(patch)
        valid_keypoints.append((y, x))

    return np.array(patches), np.array(valid_keypoints)

In [ ]:
patch_size = 11

patches1, valid_keypoints1 = extract_patches(
    gray1, keypoints1, patch_size
)

patches2, valid_keypoints2 = extract_patches(
    gray2, keypoints2, patch_size
)

print("Image 1 patches:", len(patches1))
print("Image 2 patches:", len(patches2))

print("Patch shape:", patches1[0].shape)

### Example patches

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):

    ax.imshow(patches1[i], cmap="gray")
    ax.set_title(f"Patch {i+1}")
    ax.axis("off")

plt.suptitle("Extracted patches from Image 1")
plt.tight_layout()
plt.show()

## SIFT descriptors

In [ ]:
# Create 8-bit grayscale images for SIFT
gray1_sift = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
gray2_sift = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)

print(gray1_sift.dtype)
print(gray2_sift.dtype)

### Convert keypoint coordinates

In [ ]:
def convert_to_cv_keypoints(keypoints, size=16):

    cv_keypoints = []

    for y, x in keypoints:

        kp = cv2.KeyPoint(
            x=float(x),
            y=float(y),
            size=float(size),
            angle=0
        )

        cv_keypoints.append(kp)

    return cv_keypoints

### Compute SIFT descriptors at Harris points

In [ ]:
# Create SIFT object
sift = cv2.SIFT_create()

# Convert Harris coordinates into OpenCV keypoints
cv_keypoints1 = convert_to_cv_keypoints(valid_keypoints1)
cv_keypoints2 = convert_to_cv_keypoints(valid_keypoints2)

# Compute SIFT descriptors at Harris keypoints
cv_keypoints1, descriptors1 = sift.compute(
    gray1_sift, cv_keypoints1
)

cv_keypoints2, descriptors2 = sift.compute(
    gray2_sift, cv_keypoints2
)

In [ ]:
print("Image 1:")
print("Number of SIFT keypoints:", len(cv_keypoints1))
print("Descriptor shape:", descriptors1.shape)

print("\nImage 2:")
print("Number of SIFT keypoints:", len(cv_keypoints2))
print("Descriptor shape:", descriptors2.shape)

### Normalize descriptors

In [ ]:
# Convert descriptors to float32
descriptors1 = descriptors1.astype(np.float32)
descriptors2 = descriptors2.astype(np.float32)

# Normalize descriptors for Euclidean distance
desc1_norm = descriptors1 / (
    np.linalg.norm(descriptors1, axis=1, keepdims=True) + 1e-10
)

desc2_norm = descriptors2 / (
    np.linalg.norm(descriptors2, axis=1, keepdims=True) + 1e-10
)

print("Normalized descriptor shape:", desc1_norm.shape)

In [ ]:
print("Image 1 descriptor shape:", descriptors1.shape)
print("Image 2 descriptor shape:", descriptors2.shape)

print("Image 1 descriptor length:", descriptors1.shape[1])
print("Image 2 descriptor length:", descriptors2.shape[1])

### Euclidean distance

In [ ]:
# Calculate pairwise dot products
similarities = desc1_norm @ desc2_norm.T

# Convert similarities into Euclidean distances
euclidean_distances = np.sqrt(
    np.maximum(2 - 2 * similarities, 0)
)

print("Euclidean distance matrix:", euclidean_distances.shape)

### Normalized cross-correlation (NCC)

In [ ]:
# Mean-center each descriptor
desc1_centered = descriptors1 - descriptors1.mean(axis=1, keepdims=True)
desc2_centered = descriptors2 - descriptors2.mean(axis=1, keepdims=True)

# Normalize the centered descriptors
desc1_ncc = desc1_centered / (
    np.linalg.norm(desc1_centered, axis=1, keepdims=True) + 1e-10
)

desc2_ncc = desc2_centered / (
    np.linalg.norm(desc2_centered, axis=1, keepdims=True) + 1e-10
)

# Calculate all pairwise NCC similarities
ncc_scores = desc1_ncc @ desc2_ncc.T

print("NCC matrix:", ncc_scores.shape)

### Compare descriptor scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(euclidean_distances, cmap="viridis", aspect="auto")
axes[0].set_title("Euclidean Distance")
axes[0].set_xlabel("Image 2 descriptors")
axes[0].set_ylabel("Image 1 descriptors")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(ncc_scores, cmap="viridis", aspect="auto")
axes[1].set_title("NCC Similarity")
axes[1].set_xlabel("Image 2 descriptors")
axes[1].set_ylabel("Image 1 descriptors")
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

### Select the top 300 Euclidean matches

In [ ]:
# Number of candidate matches to retain
top_n = 300

# Find the best Image 2 match for every Image 1 descriptor
best_euclidean_indices = np.argmin(
    euclidean_distances, axis=1
)

# Retrieve the corresponding distances
best_euclidean_values = euclidean_distances[
    np.arange(len(best_euclidean_indices)),
    best_euclidean_indices
]

# Rank candidates by ascending distance
euclidean_order = np.argsort(best_euclidean_values)

# Keep the top N candidates
selected_euclidean = euclidean_order[:top_n]

# Store (Image 1 index, Image 2 index) pairs
matches_euclidean = np.column_stack((
    selected_euclidean,
    best_euclidean_indices[selected_euclidean]
))

print("Euclidean matches:", len(matches_euclidean))

### Select the top 300 NCC matches

In [ ]:
# Find the best Image 2 match for every Image 1 descriptor
best_ncc_indices = np.argmax(
    ncc_scores, axis=1
)

# Retrieve the corresponding NCC scores
best_ncc_values = ncc_scores[
    np.arange(len(best_ncc_indices)),
    best_ncc_indices
]

# Rank candidates by descending similarity
ncc_order = np.argsort(best_ncc_values)[::-1]

# Keep the top N candidates
selected_ncc = ncc_order[:top_n]

# Store (Image 1 index, Image 2 index) pairs
matches_ncc = np.column_stack((
    selected_ncc,
    best_ncc_indices[selected_ncc]
))

print("NCC matches:", len(matches_ncc))

### View the matches

In [ ]:
def visualize_matches(
    image1, image2,
    keypoints1, keypoints2,
    matches,
    title
):
    h1, w1 = image1.shape[:2]
    h2, w2 = image2.shape[:2]

    # Create side-by-side canvas
    canvas = np.zeros((max(h1, h2), w1 + w2, 3), dtype=np.uint8)
    canvas[:h1, :w1] = image1
    canvas[:h2, w1:w1 + w2] = image2

    fig, ax = plt.subplots(figsize=(24, 12))
    ax.imshow(canvas)

    # Give every match a different colour
    colors = plt.cm.hsv(np.linspace(0, 1, len(matches)))

    for (idx1, idx2), color in zip(matches, colors):

        kp1 = keypoints1[idx1]
        kp2 = keypoints2[idx2]

        x1, y1 = kp1.pt
        x2, y2 = kp2.pt

        # Account for the second image's position on the canvas
        x2 += w1

        # Draw thicker connecting line
        ax.plot(
            [x1, x2], [y1, y2],
            color=color,
            linewidth=2.0,
            alpha=0.9
        )

        # Draw larger keypoint markers
        ax.scatter(
            [x1, x2], [y1, y2],
            color=[color],
            s=35,
            edgecolors="white",
            linewidths=0.8,
            zorder=3
        )

    ax.set_title(
        f"{title} — All {len(matches)} matches",
        fontsize=18
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_matches(
    image1_rgb, image2_rgb,
    cv_keypoints1, cv_keypoints2,
    matches_euclidean,
    "Euclidean Distance Matching"
)

visualize_matches(
    image1_rgb, image2_rgb,
    cv_keypoints1, cv_keypoints2,
    matches_ncc,
    "Normalized Cross-Correlation Matching"
)

### Convert matches to image coordinates

In [ ]:
def get_matched_points(matches, keypoints1, keypoints2):

    points1 = []
    points2 = []

    for idx1, idx2 in matches:

        # Coordinates in image 1
        x1, y1 = keypoints1[idx1].pt

        # Coordinates in image 2
        x2, y2 = keypoints2[idx2].pt

        points1.append([x1, y1])
        points2.append([x2, y2])

    return np.array(points1, dtype=np.float32), np.array(points2, dtype=np.float32)

In [ ]:
points1_euclidean, points2_euclidean = get_matched_points(
    matches_euclidean, cv_keypoints1, cv_keypoints2
)

points1_ncc, points2_ncc = get_matched_points(
    matches_ncc, cv_keypoints1, cv_keypoints2
)

print("Euclidean points:", points1_euclidean.shape, points2_euclidean.shape)
print("NCC points:", points1_ncc.shape, points2_ncc.shape)

print("\nFirst correspondence:")
print("Image 1:", points1_euclidean[0])
print("Image 2:", points2_euclidean[0])

### Fit an affine transform

In [ ]:
def estimate_affine(src_points, dst_points):
    """
    Estimate an affine transformation mapping src_points to dst_points.
    Requires at least 3 non-collinear point correspondences.
    """

    A = []
    B = []

    for (x, y), (u, v) in zip(src_points, dst_points):

        A.append([x, y, 0, 0, 1, 0])
        A.append([0, 0, x, y, 0, 1])

        B.append(u)
        B.append(v)

    A = np.array(A, dtype=np.float64)
    B = np.array(B, dtype=np.float64)

    # Solve for the six affine parameters
    params, residuals, rank, singular_values = np.linalg.lstsq(
        A, B, rcond=None
    )

    if rank < 6:
        return None

    a, b, c, d, tx, ty = params

    M = np.array([
        [a, b, tx],
        [c, d, ty]
    ])

    return M

In [ ]:
sample_points1 = points1_euclidean[:3]
sample_points2 = points2_euclidean[:3]

M_test = estimate_affine(sample_points2, sample_points1)

print("Estimated affine transformation:")
print(M_test)

### Check the sampled correspondences

In [ ]:
def transform_points(points, M):

    points_homogeneous = np.hstack([
        points,
        np.ones((len(points), 1))
    ])

    transformed = (M @ points_homogeneous.T).T

    return transformed

def map_points(points, M):
    """Map image-2 points using either a 2x3 affine map or a 3x3 homography."""
    points = np.asarray(points, dtype=np.float32)
    if np.asarray(M).shape == (3, 3):
        return cv2.perspectiveTransform(points.reshape(1, -1, 2), M)[0]
    return transform_points(points, M)


In [ ]:
transformed_sample = transform_points(sample_points2, M_test)

print("Original points in image 1:")
print(sample_points1)

print("\nTransformed points from image 2:")
print(transformed_sample)

print("\nErrors:")
print(np.linalg.norm(sample_points1 - transformed_sample, axis=1))

### RANSAC implementation

In [ ]:
def ransac_affine(
    points1,
    points2,
    iterations=2000,
    threshold=3.0
):
    """
    Custom RANSAC affine estimator.
    Estimates a transform mapping image-2 coordinates to image-1 coordinates.
    Models are ranked first by inlier count, then by median inlier error.
    """
    points1 = np.asarray(points1, dtype=np.float32)
    points2 = np.asarray(points2, dtype=np.float32)

    if len(points1) != len(points2):
        raise ValueError("The two point arrays must have equal lengths.")
    if len(points1) < 3:
        raise ValueError("At least 3 matches are required.")

    best_M = None
    best_inliers = None
    best_count = 0
    best_median_error = np.inf

    for _ in range(iterations):
        sample_indices = np.random.choice(len(points1), 3, replace=False)
        sample1 = points1[sample_indices]
        sample2 = points2[sample_indices]

        # Reject nearly collinear samples; they produce unstable affine maps.
        if abs(cv2.contourArea(cv2.convexHull(sample2.reshape(-1, 1, 2)))) < 1.0:
            continue
        if abs(cv2.contourArea(cv2.convexHull(sample1.reshape(-1, 1, 2)))) < 1.0:
            continue

        M = estimate_affine(sample2, sample1)
        if M is None or not np.isfinite(M).all():
            continue

        linear_det = np.linalg.det(M[:, :2])
        if not np.isfinite(linear_det) or abs(linear_det) < 1e-8:
            continue

        predicted = transform_points(points2, M)
        errors = np.linalg.norm(points1 - predicted, axis=1)
        inliers = errors < threshold
        count = int(np.count_nonzero(inliers))

        if count < 3:
            continue

        median_error = float(np.median(errors[inliers]))
        if (count > best_count) or (
            count == best_count and median_error < best_median_error
        ):
            best_M = M
            best_inliers = inliers
            best_count = count
            best_median_error = median_error

    return best_M, best_inliers


In [ ]:
M_euclidean, inliers_euclidean = ransac_affine(
    points1_euclidean,
    points2_euclidean
)

M_ncc, inliers_ncc = ransac_affine(
    points1_ncc,
    points2_ncc
)

print("EUCLIDEAN MATCHING")
print("Inliers:", np.sum(inliers_euclidean))
print("Outliers:", len(inliers_euclidean) - np.sum(inliers_euclidean))
print("Inlier ratio:", np.mean(inliers_euclidean))

print("\nNCC MATCHING")
print("Inliers:", np.sum(inliers_ncc))
print("Outliers:", len(inliers_ncc) - np.sum(inliers_ncc))
print("Inlier ratio:", np.mean(inliers_ncc))

### View RANSAC inliers and outliers

In [ ]:
def visualize_ransac_matches(
    image1, image2,
    keypoints1, keypoints2,
    matches, inliers,
    title
):
    h1, w1 = image1.shape[:2]
    h2, w2 = image2.shape[:2]

    canvas = np.zeros(
        (max(h1, h2), w1 + w2, 3),
        dtype=np.uint8
    )

    canvas[:h1, :w1] = image1
    canvas[:h2, w1:w1 + w2] = image2

    fig, ax = plt.subplots(figsize=(24, 12))
    ax.imshow(canvas)

    for (idx1, idx2), is_inlier in zip(matches, inliers):

        kp1 = keypoints1[idx1]
        kp2 = keypoints2[idx2]

        x1, y1 = kp1.pt
        x2, y2 = kp2.pt
        x2 += w1

        if is_inlier:
            color = "lime"
            alpha = 0.8
            linewidth = 1.8
        else:
            color = "red"
            alpha = 0.8
            linewidth = 1.5

        ax.plot(
            [x1, x2], [y1, y2],
            color=color,
            linewidth=linewidth,
            alpha=alpha
        )

        ax.scatter(
            [x1, x2], [y1, y2],
            color=color,
            s=20,
            zorder=3
        )

    ax.set_title(title, fontsize=18)
    ax.axis("off")

    from matplotlib.lines import Line2D

    legend = [
        Line2D([0], [0], color="lime", lw=3, label="Inliers"),
        Line2D([0], [0], color="red", lw=3, label="Outliers")
    ]

    ax.legend(handles=legend, loc="upper right", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_ransac_matches(
    image1_rgb, image2_rgb,
    cv_keypoints1, cv_keypoints2,
    matches_euclidean, inliers_euclidean,
    "RANSAC — Euclidean Matching"
)

visualize_ransac_matches(
    image1_rgb, image2_rgb,
    cv_keypoints1, cv_keypoints2,
    matches_ncc, inliers_ncc,
    "RANSAC — NCC Matching"
)

### Refine the transform with its inliers

In [ ]:
def refine_affine(points1, points2, inliers):

    # Keep only geometrically consistent correspondences
    inlier_points1 = points1[inliers]
    inlier_points2 = points2[inliers]

    # Re-estimate transformation using all inliers
    refined_M = estimate_affine(
        inlier_points2,
        inlier_points1
    )

    return refined_M

In [ ]:
def refine_and_evaluate(points1, points2, M, inliers, threshold=3.0):
    """
    Iteratively refit the affine transform using geometrically consistent
    correspondences, then return final residual metrics and inlier mask.
    """
    if M is None or inliers is None or np.count_nonzero(inliers) < 3:
        raise ValueError("Cannot refine an affine transform without 3 inliers.")

    current_inliers = np.asarray(inliers, dtype=bool).copy()
    M_refined = M

    for _ in range(5):
        if np.count_nonzero(current_inliers) < 3:
            break

        candidate = estimate_affine(
            points2[current_inliers],
            points1[current_inliers]
        )
        if candidate is None or not np.isfinite(candidate).all():
            break

        M_refined = candidate
        predicted = transform_points(points2, M_refined)
        errors = np.linalg.norm(points1 - predicted, axis=1)
        updated_inliers = errors < threshold

        if np.array_equal(updated_inliers, current_inliers):
            current_inliers = updated_inliers
            break
        current_inliers = updated_inliers

    if np.count_nonzero(current_inliers) < 3:
        raise ValueError("Affine refinement left fewer than 3 inliers.")

    # Final least-squares fit on the stable inlier set.
    final_M = estimate_affine(
        points2[current_inliers],
        points1[current_inliers]
    )
    if final_M is not None and np.isfinite(final_M).all():
        M_refined = final_M

    final_predicted = transform_points(points2, M_refined)
    final_errors = np.linalg.norm(points1 - final_predicted, axis=1)
    final_inliers = final_errors < threshold

    if np.count_nonzero(final_inliers) < 3:
        raise ValueError("Final affine model has fewer than 3 inliers.")

    inlier_errors = final_errors[final_inliers]
    mean_error = float(np.mean(inlier_errors))
    mse = float(np.mean(inlier_errors ** 2))

    return M_refined, final_inliers, mean_error, mse


## Prepare patch descriptors

In [ ]:
patch_size = 11

patches1, valid_keypoints1 = extract_patches(
    gray1, keypoints1, patch_size
)

patches2, valid_keypoints2 = extract_patches(
    gray2, keypoints2, patch_size
)

print("Image 1 patches:", patches1.shape)
print("Image 2 patches:", patches2.shape)

### Flatten patches into descriptors

In [ ]:
patch_descriptors1 = patches1.reshape(len(patches1), -1)
patch_descriptors2 = patches2.reshape(len(patches2), -1)

print("Descriptor 1:", patch_descriptors1.shape)
print("Descriptor 2:", patch_descriptors2.shape)

### Normalize patch descriptors

In [ ]:
patch_desc1_norm = patch_descriptors1 / (
    np.linalg.norm(patch_descriptors1, axis=1, keepdims=True) + 1e-10
)

patch_desc2_norm = patch_descriptors2 / (
    np.linalg.norm(patch_descriptors2, axis=1, keepdims=True) + 1e-10
)
print("Descriptor 1 normalized:", patch_desc1_norm.shape)
print("Descriptor 2 normalized:", patch_desc2_norm.shape)

### Calculate patch Euclidean distances

In [ ]:
patch_similarities = patch_desc1_norm @ patch_desc2_norm.T

patch_euclidean_distances = np.sqrt(
    np.maximum(2 - 2 * patch_similarities, 0)
)

print("Patch Euclidean matrix:", patch_euclidean_distances.shape)

### Calculate patch NCC scores

In [ ]:
patch_desc1_centered = (
    patch_descriptors1 -
    patch_descriptors1.mean(axis=1, keepdims=True)
)

patch_desc2_centered = (
    patch_descriptors2 -
    patch_descriptors2.mean(axis=1, keepdims=True)
)

patch_desc1_ncc = patch_desc1_centered / (
    np.linalg.norm(patch_desc1_centered, axis=1, keepdims=True) + 1e-10
)

patch_desc2_ncc = patch_desc2_centered / (
    np.linalg.norm(patch_desc2_centered, axis=1, keepdims=True) + 1e-10
)

patch_ncc_scores = patch_desc1_ncc @ patch_desc2_ncc.T

print("Patch NCC matrix:", patch_ncc_scores.shape)

### Inspect patch matching scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im1 = axes[0].imshow(patch_euclidean_distances, aspect="auto")
axes[0].set_title("Patch Euclidean Distances")
axes[0].set_xlabel("Image 2 descriptors")
axes[0].set_ylabel("Image 1 descriptors")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(patch_ncc_scores, aspect="auto", cmap="coolwarm")
axes[1].set_title("Patch NCC Scores")
axes[1].set_xlabel("Image 2 descriptors")
axes[1].set_ylabel("Image 1 descriptors")
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

## Match patch descriptors

### Euclidean matching

In [ ]:
top_n = 300

best_patch_euclidean_indices = np.argmin(
    patch_euclidean_distances, axis=1
)

best_patch_euclidean_values = patch_euclidean_distances[
    np.arange(len(best_patch_euclidean_indices)),
    best_patch_euclidean_indices
]

patch_euclidean_order = np.argsort(best_patch_euclidean_values)

selected_patch_euclidean = patch_euclidean_order[:top_n]

matches_patch_euclidean = np.column_stack((
    selected_patch_euclidean,
    best_patch_euclidean_indices[selected_patch_euclidean]
))

### NCC matching

In [ ]:
best_patch_ncc_indices = np.argmax(
    patch_ncc_scores, axis=1
)

best_patch_ncc_values = patch_ncc_scores[
    np.arange(len(best_patch_ncc_indices)),
    best_patch_ncc_indices
]

patch_ncc_order = np.argsort(best_patch_ncc_values)[::-1]

selected_patch_ncc = patch_ncc_order[:top_n]

matches_patch_ncc = np.column_stack((
    selected_patch_ncc,
    best_patch_ncc_indices[selected_patch_ncc]
))

In [ ]:
points1_patch, points2_patch = get_matched_points(
    matches_patch_euclidean,
    convert_to_cv_keypoints(valid_keypoints1),
    convert_to_cv_keypoints(valid_keypoints2)
)

points1_patch_ncc, points2_patch_ncc = get_matched_points(
    matches_patch_ncc,
    convert_to_cv_keypoints(valid_keypoints1),
    convert_to_cv_keypoints(valid_keypoints2)
)
print("Patch Euclidean:", points1_patch.shape, points2_patch.shape)
print("Patch NCC:", points1_patch_ncc.shape, points2_patch_ncc.shape)

### Estimate transforms from patch matches

In [ ]:
M_patch_euclidean, inliers_patch_euclidean = ransac_affine(
    points1_patch,
    points2_patch,
    iterations=1000,
    threshold=5
)

M_patch_ncc, inliers_patch_ncc = ransac_affine(
    points1_patch_ncc,
    points2_patch_ncc,
    iterations=1000,
    threshold=5
)

### Refine and compare the transforms

In [ ]:
# Patch Euclidean
M_patch_euclidean_refined, inliers_patch_euclidean, patch_euclidean_mean_error, patch_euclidean_mse = refine_and_evaluate(
    points1_patch, points2_patch,
    M_patch_euclidean, inliers_patch_euclidean
)

# Patch NCC
M_patch_ncc_refined, inliers_patch_ncc, patch_ncc_mean_error, patch_ncc_mse = refine_and_evaluate(
    points1_patch_ncc, points2_patch_ncc,
    M_patch_ncc, inliers_patch_ncc
)

# SIFT Euclidean
M_euclidean_refined, inliers_euclidean, sift_euclidean_mean_error, sift_euclidean_mse = refine_and_evaluate(
    points1_euclidean, points2_euclidean,
    M_euclidean, inliers_euclidean
)

# SIFT NCC
M_ncc_refined, inliers_ncc, sift_ncc_mean_error, sift_ncc_mse = refine_and_evaluate(
    points1_ncc, points2_ncc,
    M_ncc, inliers_ncc
)

method_comparison_results = {
    "Patch Euclidean": (inliers_patch_euclidean, patch_euclidean_mean_error, patch_euclidean_mse),
    "Patch NCC": (inliers_patch_ncc, patch_ncc_mean_error, patch_ncc_mse),
    "SIFT Euclidean": (inliers_euclidean, sift_euclidean_mean_error, sift_euclidean_mse),
    "SIFT NCC": (inliers_ncc, sift_ncc_mean_error, sift_ncc_mse)
}

for method, (inliers, mean_error, mse) in method_comparison_results.items():
    print(f"\n{method}")
    print("Inliers:", np.sum(inliers))
    print("Outliers:", len(inliers) - np.sum(inliers))
    print("Mean residual:", round(mean_error, 4))
    print("MSE:", round(mse, 4))

### Measure reprojection error

In [ ]:
def calculate_residuals(points1, points2, M, inliers):

    # Transform image 2 coordinates into image 1 coordinates
    transformed_points = transform_points(points2, M)

    # Keep only inlier errors
    errors = np.linalg.norm(
        points1[inliers] - transformed_points[inliers],
        axis=1
    )

    mean_error = np.mean(errors)
    mean_squared_error = np.mean(errors ** 2)

    return mean_error, mean_squared_error

In [ ]:
euclidean_mean_error, euclidean_mse = calculate_residuals(
    points1_euclidean,
    points2_euclidean,
    M_euclidean_refined,
    inliers_euclidean
)

ncc_mean_error, ncc_mse = calculate_residuals(
    points1_ncc,
    points2_ncc,
    M_ncc_refined,
    inliers_ncc
)

print("EUCLIDEAN")
print("Mean residual:", euclidean_mean_error, "pixels")
print("Mean squared residual:", euclidean_mse, "pixels²")

print("\nNCC")
print("Mean residual:", ncc_mean_error, "pixels")
print("Mean squared residual:", ncc_mse, "pixels²")

## Create Pair 1 panoramas

In [ ]:
def create_panorama(image1, image2, M):

    h1, w1 = image1.shape[:2]
    h2, w2 = image2.shape[:2]

    # Corners of image 1
    corners1 = np.array([
        [0, 0],
        [w1, 0],
        [w1, h1],
        [0, h1]
    ], dtype=np.float32)

    # Corners of image 2
    corners2 = np.array([
        [0, 0],
        [w2, 0],
        [w2, h2],
        [0, h2]
    ], dtype=np.float32)

    # Transform image 2 corners into image 1 coordinates
    transformed_corners2 = transform_points(corners2, M)

    all_corners = np.vstack([corners1, transformed_corners2])

    min_x, min_y = np.floor(all_corners.min(axis=0)).astype(int)
    max_x, max_y = np.ceil(all_corners.max(axis=0)).astype(int)

    # Create 3x3 translation matrix
    T = np.array([
        [1, 0, -min_x],
        [0, 1, -min_y],
        [0, 0, 1]
    ], dtype=np.float64)

    # Convert affine matrix to homogeneous 3x3 format
    M_homogeneous = np.vstack([
        M,
        [0, 0, 1]
    ])

    # Combine translation and affine transformation
    final_transform = T @ M_homogeneous

    output_width = max_x - min_x
    output_height = max_y - min_y

    # Warp image 2
    panorama = cv2.warpAffine(
        image2,
        final_transform[:2, :],
        (output_width, output_height)
    )

    # Place image 1 on the canvas
    x_offset = -min_x
    y_offset = -min_y

    panorama[
        y_offset:y_offset + h1,
        x_offset:x_offset + w1
    ] = image1

    return panorama

In [ ]:
panorama_euclidean = create_panorama(
    image1,
    image2,
    M_euclidean_refined
)

panorama_ncc = create_panorama(
    image1,
    image2,
    M_ncc_refined
)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

axes[0].imshow(cv2.cvtColor(panorama_euclidean, cv2.COLOR_BGR2RGB))
axes[0].set_title("Euclidean Matching Panorama")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(panorama_ncc, cv2.COLOR_BGR2RGB))
axes[1].set_title("NCC Matching Panorama")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### Blend the warped images

In [ ]:
def create_blended_panorama(image1, image2, M, feather_width=18, max_canvas_pixels=80000000):
    """Warp image 2 and blend with contained-image overlay or a narrow seam.

    If one image is almost entirely contained in the other, keep the smaller
    image on top throughout its footprint and feather only its boundary. For
    side-by-side panoramas, blend only in a narrow band around the equal-
    distance seam so residual alignment error does not create doubled edges.
    """
    h1, w1 = image1.shape[:2]
    h2, w2 = image2.shape[:2]
    M = np.asarray(M, dtype=np.float64)
    if M.shape != (2, 3) or not np.isfinite(M).all():
        raise ValueError("Expected a finite 2x3 affine matrix.")
    c1 = np.array([[0,0],[w1,0],[w1,h1],[0,h1]], np.float32)
    c2 = np.array([[0,0],[w2,0],[w2,h2],[0,h2]], np.float32)
    c2 = transform_points(c2, M)
    corners = np.vstack([c1, c2])
    min_x, min_y = np.floor(corners.min(axis=0)).astype(int)
    max_x, max_y = np.ceil(corners.max(axis=0)).astype(int)
    W, H = int(max_x-min_x), int(max_y-min_y)
    if W <= 0 or H <= 0 or W*H > max_canvas_pixels:
        raise ValueError(f"Invalid panorama canvas {W}x{H}.")
    T = np.array([[1,0,-min_x],[0,1,-min_y],[0,0,1]], np.float64)
    final = (T @ np.vstack([M,[0,0,1]]))[:2]
    warped2 = cv2.warpAffine(image2, final, (W,H), flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    mask2 = cv2.warpAffine(np.full((h2,w2),255,np.uint8), final, (W,H),
                           flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT) > 0
    ox, oy = -min_x, -min_y
    mask1_u8 = np.zeros((H,W), np.uint8)
    mask1_u8[oy:oy+h1, ox:ox+w1] = 255
    valid1 = mask1_u8 > 0
    area1, area2 = int(np.count_nonzero(valid1)), int(np.count_nonzero(mask2))
    overlap = int(np.count_nonzero(valid1 & mask2))
    smaller, larger = min(area1,area2), max(area1,area2)
    contained = overlap / max(smaller,1) >= 0.98 and smaller / max(larger,1) < 0.85
    feather = max(float(feather_width),1.0)
    canvas1 = np.zeros_like(warped2)
    canvas1[oy:oy+h1, ox:ox+w1] = image1

    if contained:
        # The smaller view is the foreground layer; blend it only at its edge.
        small_is_1 = area1 <= area2
        small_mask = valid1 if small_is_1 else mask2
        small_rgb = canvas1 if small_is_1 else warped2
        large_rgb = warped2 if small_is_1 else canvas1
        small_distance = cv2.distanceTransform(small_mask.astype(np.uint8), cv2.DIST_L2, 3)
        alpha_small = np.clip(small_distance / feather, 0.0, 1.0)
        alpha_small[small_mask & ~ (mask2 if small_is_1 else valid1)] = 1.0
        alpha_small[~small_mask] = 0.0
        result = (small_rgb.astype(np.float32)*alpha_small[:,:,None]
                  + large_rgb.astype(np.float32)*(1.0-alpha_small[:,:,None]))
    else:
        # Non-contained panoramas: make a narrow transition around the seam.
        d1 = cv2.distanceTransform(mask1_u8, cv2.DIST_L2, 3)
        d2 = cv2.distanceTransform(mask2.astype(np.uint8), cv2.DIST_L2, 3)
        weight1 = np.clip(0.5+(d1-d2)/(2.0*feather), 0.0, 1.0)
        weight1[valid1 & ~mask2] = 1.0
        weight1[~valid1 & mask2] = 0.0
        result = (canvas1.astype(np.float32)*weight1[:,:,None]
                  + warped2.astype(np.float32)*(1.0-weight1[:,:,None]))
    return np.clip(result,0,255).astype(np.uint8)


In [ ]:
blended_euclidean = create_blended_panorama(
    image1, image2, M_euclidean_refined
)

blended_ncc = create_blended_panorama(
    image1, image2, M_ncc_refined
)

fig, axes = plt.subplots(2, 2, figsize=(20, 12))

axes[0, 0].imshow(cv2.cvtColor(panorama_euclidean, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("Euclidean - Simple Overlay")

axes[0, 1].imshow(cv2.cvtColor(blended_euclidean, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title("Euclidean - Feather Blending")

axes[1, 0].imshow(cv2.cvtColor(panorama_ncc, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("NCC - Simple Overlay")

axes[1, 1].imshow(cv2.cvtColor(blended_ncc, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("NCC - Feather Blending")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Patch-size sensitivity

In [ ]:
def evaluate_patch_size(patch_size, seed=42):

    patches1, kp1 = extract_patches(
        gray1, keypoints1, patch_size
    )

    patches2, kp2 = extract_patches(
        gray2, keypoints2, patch_size
    )

    # Flatten patches into descriptors
    desc1 = patches1.reshape(len(patches1), -1).astype(np.float32)
    desc2 = patches2.reshape(len(patches2), -1).astype(np.float32)

    # L2 normalization
    desc1_norm = desc1 / (
        np.linalg.norm(desc1, axis=1, keepdims=True) + 1e-10
    )
    desc2_norm = desc2 / (
        np.linalg.norm(desc2, axis=1, keepdims=True) + 1e-10
    )

    # Euclidean distances
    similarities = desc1_norm @ desc2_norm.T
    euclidean_distances = np.sqrt(
        np.maximum(2 - 2 * similarities, 0)
    )

    # NCC scores
    desc1_centered = desc1 - desc1.mean(axis=1, keepdims=True)
    desc2_centered = desc2 - desc2.mean(axis=1, keepdims=True)

    desc1_ncc = desc1_centered / (
        np.linalg.norm(desc1_centered, axis=1, keepdims=True) + 1e-10
    )
    desc2_ncc = desc2_centered / (
        np.linalg.norm(desc2_centered, axis=1, keepdims=True) + 1e-10
    )

    ncc_scores = desc1_ncc @ desc2_ncc.T

    results = {}

    for method, matrix, mode in [
        ("Euclidean", euclidean_distances, "min"),
        ("NCC", ncc_scores, "max")
    ]:

        best_indices = (
            np.argmin(matrix, axis=1)
            if mode == "min"
            else np.argmax(matrix, axis=1)
        )

        best_values = matrix[np.arange(len(best_indices)), best_indices]

        order = (
            np.argsort(best_values)
            if mode == "min"
            else np.argsort(best_values)[::-1]
        )

        selected = order[:300]

        matches = np.column_stack((
            selected,
            best_indices[selected]
        ))

        # Convert (y, x) coordinates into OpenCV keypoints
        cv_kp1 = convert_to_cv_keypoints(kp1)
        cv_kp2 = convert_to_cv_keypoints(kp2)

        points1, points2 = get_matched_points(
            matches, cv_kp1, cv_kp2
        )

        # Reproducible RANSAC run
        np.random.seed(seed)

        M, inliers = ransac_affine(
            points1, points2,
            iterations=1000,
            threshold=5
        )

        if M is None or inliers is None or np.sum(inliers) < 3:
            results[method] = {
                "inliers": 0,
                "mean_error": np.nan,
                "mse": np.nan
            }
            continue

        M_refined = refine_affine(points1, points2, inliers)

        mean_error, mse = calculate_residuals(
            points1, points2, M_refined, inliers
        )

        results[method] = {
            "inliers": int(np.sum(inliers)),
            "mean_error": mean_error,
            "mse": mse
        }

    return results

### Compare the four patch sizes

In [ ]:
patch_sizes = [7, 11, 15, 21]

patch_sensitivity_results = {}

for size in patch_sizes:

    print(f"Testing patch size: {size}")

    patch_sensitivity_results[size] = evaluate_patch_size(size)

print("\nExperiment completed.")
for size, results in patch_sensitivity_results.items():

    print(f"\nPatch size: {size}")

    for method, metrics in results.items():

        print(
            method,
            "| Inliers:", metrics["inliers"],
            "| Mean error:", round(metrics["mean_error"], 4),
            "| MSE:", round(metrics["mse"], 4)
        )

## Matching sensitivity

I vary the number of SIFT--NCC candidates to see how it changes inlier counts and reprojection error.

In [ ]:
top_n_values = [100, 200, 300, 400, 500]

top_n_results = {}

for n in top_n_values:

    # Select the best NCC match for each descriptor in image 1
    best_indices = np.argmax(ncc_scores, axis=1)

    best_values = ncc_scores[
        np.arange(len(best_indices)),
        best_indices
    ]

    # Sort matches by NCC score (highest first)
    order = np.argsort(best_values)[::-1]

    # Select the requested number of matches
    selected = order[:n]

    matches = np.column_stack((
        selected,
        best_indices[selected]
    ))

    # Convert selected matches into point coordinates
    points1, points2 = get_matched_points(
        matches,
        cv_keypoints1,
        cv_keypoints2
    )

    # Run RANSAC
    np.random.seed(42)

    M, inliers = ransac_affine(
        points1,
        points2,
        iterations=1000,
        threshold=5
    )

    # Refine and evaluate
    M_refined, final_inliers, mean_error, mse = refine_and_evaluate(
        points1,
        points2,
        M,
        inliers,
        threshold=5
    )

    top_n_results[n] = {
        "inliers": int(np.sum(final_inliers)),
        "outliers": int(len(final_inliers) - np.sum(final_inliers)),
        "mean_error": mean_error,
        "mse": mse
    }

    print(
        f"Top {n}: "
        f"Inliers={top_n_results[n]['inliers']}, "
        f"Mean error={mean_error:.4f}, "
        f"MSE={mse:.4f}"
    )

## RANSAC threshold sensitivity

In [ ]:
ransac_thresholds = [1, 2, 3, 5, 8, 10]

threshold_results = {}

#Same 300 SIFT-NCC matches for every experiment
best_indices = np.argmax(ncc_scores, axis=1)

best_values = ncc_scores[
    np.arange(len(best_indices)),
    best_indices
]

order = np.argsort(best_values)[::-1]
selected = order[:300]

matches = np.column_stack((selected, best_indices[selected]))

points1, points2 = get_matched_points(
    matches,
    cv_keypoints1,
    cv_keypoints2
)

for threshold in ransac_thresholds:

    np.random.seed(42)

    M, inliers = ransac_affine(
        points1,
        points2,
        iterations=1000,
        threshold=threshold
    )

    M_refined, final_inliers, mean_error, mse = refine_and_evaluate(
        points1,
        points2,
        M,
        inliers,
        threshold=threshold
    )

    threshold_results[threshold] = {
        "inliers": int(np.sum(final_inliers)),
        "outliers": int(len(final_inliers) - np.sum(final_inliers)),
        "mean_error": mean_error,
        "mse": mse
    }

    print(
        f"Threshold {threshold}: "
        f"Inliers={threshold_results[threshold]['inliers']}, "
        f"Mean error={mean_error:.4f}, "
        f"MSE={mse:.4f}"
    )

## RANSAC iteration sensitivity

In [ ]:
ransac_iterations = [50, 100, 250, 500, 1000, 2000]

iteration_results = {}

# Reuse the same 300 SIFT-NCC matches
# and the same threshold of 5 pixels.

for iterations in ransac_iterations:

    np.random.seed(42)

    M, inliers = ransac_affine(
        points1,
        points2,
        iterations=iterations,
        threshold=5
    )

    M_refined, final_inliers, mean_error, mse = refine_and_evaluate(
        points1,
        points2,
        M,
        inliers,
        threshold=5
    )

    iteration_results[iterations] = {
        "inliers": int(np.sum(final_inliers)),
        "mean_error": mean_error,
        "mse": mse
    }

    print(
        f"Iterations {iterations}: "
        f"Inliers={iteration_results[iterations]['inliers']}, "
        f"Mean error={mean_error:.4f}, "
        f"MSE={mse:.4f}"
    )

## Random-seed sensitivity

In [ ]:
random_seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

stability_results = {}

for seed in random_seeds:

    np.random.seed(seed)

    M, inliers = ransac_affine(
        points1,
        points2,
        iterations=50,
        threshold=5
    )

    M_refined, final_inliers, mean_error, mse = refine_and_evaluate(
        points1,
        points2,
        M,
        inliers,
        threshold=5
    )

    stability_results[seed] = {
        "inliers": int(np.sum(final_inliers)),
        "mean_error": mean_error,
        "mse": mse
    }

    print(
        f"Seed {seed}: "
        f"Inliers={stability_results[seed]['inliers']}, "
        f"Mean error={mean_error:.4f}, "
        f"MSE={mse:.4f}"
    )

In [ ]:
# Show every sensitivity sweep in one figure: rows are experiments and columns are metrics.
sensitivity_specs = [
    ("Patch size", list(patch_sensitivity_results.keys()), {
        "Euclidean": patch_sensitivity_results,
        "NCC": patch_sensitivity_results
    }, True),
    ("Candidate matches", list(top_n_results.keys()), {"SIFT NCC": top_n_results}, False),
    ("RANSAC threshold (px)", list(threshold_results.keys()), {"SIFT NCC": threshold_results}, False),
    ("RANSAC iterations", list(iteration_results.keys()), {"SIFT NCC": iteration_results}, False),
    ("Random seed", list(stability_results.keys()), {"SIFT NCC": stability_results}, False)
]
metric_specs = [("inliers", "Inliers"), ("mean_error", "Mean residual (px)"), ("mse", "MSE (px²)")]
fig, axes = plt.subplots(len(sensitivity_specs), 3, figsize=(16, 19))
for row, (experiment, x_values, series, nested) in enumerate(sensitivity_specs):
    for col, (metric, metric_label) in enumerate(metric_specs):
        ax = axes[row, col]
        if nested:
            for method in ("Euclidean", "NCC"):
                y_values = [series[method][x][method][metric] for x in x_values]
                ax.plot(x_values, y_values, marker="o", label=method)
            if row == 0 and col == 0:
                ax.legend()
        else:
            data = next(iter(series.values()))
            y_values = [data[x][metric] for x in x_values]
            ax.plot(x_values, y_values, marker="o", color="tab:blue")
        ax.set_title(metric_label if row == 0 else "")
        ax.set_xlabel(experiment)
        ax.set_ylabel(metric_label)
        ax.grid(True, alpha=0.3)
fig.suptitle("Consolidated sensitivity analysis", fontsize=16, y=1.002)
fig.tight_layout()
fig.savefig(OUTPUT_FOLDER / "sensitivity_summary.png", dpi=130, bbox_inches="tight")
plt.show()


## Load the five image pairs

In [ ]:
IMAGE_FOLDER = Path("images")
image_pairs = {
    "Pair 1": ("leftImage.png", "rightImage.png"),
    "Pair 2": ("ul1.png", "ur1.png"),
    "Pair 3": ("past1.png", "past2.png"),
    "Pair 4": ("cr1.jpg", "cr2.jpg"),
    "Pair 5": ("shi1.jpeg", "shi2.jpeg")
}

In [ ]:
def load_image_pair(pair_name):

    filename1, filename2 = image_pairs[pair_name]

    image1_path = IMAGE_FOLDER / filename1
    image2_path = IMAGE_FOLDER / filename2

    image1 = cv2.imread(str(image1_path))
    image2 = cv2.imread(str(image2_path))

    if image1 is None or image2 is None:
        raise FileNotFoundError(
            f"Could not load images for {pair_name}"
        )

    return image1, image2

In [ ]:
for pair_name in image_pairs:

    image1, image2 = load_image_pair(pair_name)

    print(f"\n{pair_name}")
    print("Image 1 dimensions:", image1.shape)
    print("Image 2 dimensions:", image2.shape)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(cv2.cvtColor(image1, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"{pair_name} - Image 1")
    axes[0].axis("off")

    axes[1].imshow(cv2.cvtColor(image2, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"{pair_name} - Image 2")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

## Stitching pipeline

This function runs the Harris-to-panorama pipeline. It filters matches, estimates and refines the affine transform, reports inlier coverage and reprojection error, then blends the images. Weak spatial coverage is reported so the remaining pairs can still be processed; invalid transforms or excessive reprojection error stop the blend.

In [ ]:
def summarize_inlier_spread(points, image_shape, grid_size=4):
    """Measure whether inliers cover a useful portion of one image."""
    height, width = image_shape[:2]
    points = np.asarray(points, dtype=np.float32)
    if len(points) < 3:
        return {"hull_fraction": 0.0, "occupied_cells": 0, "point_count": len(points)}

    grid_xy = np.floor(points / np.array([width, height]) * grid_size).astype(int)
    grid_xy = np.clip(grid_xy, 0, grid_size - 1)
    occupied_cells = len({(int(x), int(y)) for x, y in grid_xy})
    hull = cv2.convexHull(points.reshape(-1, 1, 2))
    hull_fraction = float(cv2.contourArea(hull) / (width * height))
    return {
        "hull_fraction": hull_fraction,
        "occupied_cells": occupied_cells,
        "point_count": len(points)
    }


def plot_alignment_diagnostic(results, pair_name):
    """Show inlier spread and compare matched points after the estimated warp."""
    image1_rgb = cv2.cvtColor(results["image1"], cv2.COLOR_BGR2RGB)
    image2_rgb = cv2.cvtColor(results["image2"], cv2.COLOR_BGR2RGB)
    inlier_points1 = results["points1"][results["inliers"]]
    inlier_points2 = results["points2"][results["inliers"]]
    mapped_points2 = map_points(inlier_points2, results["transformation"])
    spread1, spread2 = results["inlier_spatial"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image1_rgb)
    axes[0].scatter(inlier_points1[:, 0], inlier_points1[:, 1], s=9, c="lime", alpha=0.7)
    axes[0].set_title(f"Image 1 inliers\n{spread1['occupied_cells']}/16 cells; hull {spread1['hull_fraction']:.1%}")
    axes[0].axis("off")

    axes[1].imshow(image2_rgb)
    axes[1].scatter(inlier_points2[:, 0], inlier_points2[:, 1], s=9, c="cyan", alpha=0.7)
    axes[1].set_title(f"Image 2 inliers\n{spread2['occupied_cells']}/16 cells; hull {spread2['hull_fraction']:.1%}")
    axes[1].axis("off")

    # Corresponding green/red points should nearly coincide after mapping image 2 to image 1.
    axes[2].imshow(image1_rgb)
    axes[2].scatter(inlier_points1[:, 0], inlier_points1[:, 1], s=13, c="lime", label="Image 1 inlier")
    axes[2].scatter(mapped_points2[:, 0], mapped_points2[:, 1], s=13, c="red", marker="x", label="Mapped image 2 inlier")
    line_indices = np.linspace(0, len(inlier_points1) - 1, min(80, len(inlier_points1))).astype(int)
    for j in line_indices:
        axes[2].plot(
            [inlier_points1[j, 0], mapped_points2[j, 0]],
            [inlier_points1[j, 1], mapped_points2[j, 1]],
            color="yellow", linewidth=0.6, alpha=0.45
        )
    axes[2].set_title(
        f"Mapped inliers before blending\nmedian {results['median_inlier_error']:.2f}px; "
        f"95th percentile {results['p95_inlier_error']:.2f}px"
    )
    axes[2].legend(loc="lower right", fontsize=8)
    axes[2].axis("off")
    fig.suptitle(f"{pair_name}: {results['transform_model']} spatial support and alignment", fontsize=15)
    fig.tight_layout()
    return fig


In [ ]:
def process_image_pair(
    pair_name,
    top_n=300,
    ransac_iterations=2000,
    ransac_threshold=3.0,
    random_seed=42,
    min_ncc=0.75,
    min_ncc_gap=0.02
):
    """
    Full Harris -> SIFT -> NCC -> mutual/ambiguity filtering ->
    custom affine RANSAC -> refinement -> feather-blended panorama pipeline.
    """
    print(f"\n{'='*60}\nProcessing {pair_name}\n{'='*60}")

    image1, image2 = load_image_pair(pair_name)

    gray1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    gray2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0

    # 1. Image gradients
    Ix1 = cv2.Sobel(gray1, cv2.CV_32F, 1, 0, ksize=3)
    Iy1 = cv2.Sobel(gray1, cv2.CV_32F, 0, 1, ksize=3)
    Ix2 = cv2.Sobel(gray2, cv2.CV_32F, 1, 0, ksize=3)
    Iy2 = cv2.Sobel(gray2, cv2.CV_32F, 0, 1, ksize=3)

    # 2. Harris corner response
    window_size, sigma, alpha = 5, 1, 0.04

    def calculate_harris_response(Ix, Iy):
        Sxx = cv2.GaussianBlur(Ix * Ix, (window_size, window_size), sigma)
        Syy = cv2.GaussianBlur(Iy * Iy, (window_size, window_size), sigma)
        Sxy = cv2.GaussianBlur(Ix * Iy, (window_size, window_size), sigma)
        determinant = Sxx * Syy - Sxy * Sxy
        trace = Sxx + Syy
        return determinant - alpha * (trace ** 2)

    R1 = calculate_harris_response(Ix1, Iy1)
    R2 = calculate_harris_response(Ix2, Iy2)

    # 3. Harris keypoints
    keypoints1 = detect_keypoints(R1, threshold_ratio=0.01, nms_size=5)
    keypoints2 = detect_keypoints(R2, threshold_ratio=0.01, nms_size=5)
    print("Harris keypoints:", len(keypoints1), len(keypoints2))

    # 4. SIFT descriptors at Harris locations
    gray1_sift = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
    gray2_sift = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create()

    cv_keypoints1 = convert_to_cv_keypoints(keypoints1)
    cv_keypoints2 = convert_to_cv_keypoints(keypoints2)

    cv_keypoints1, descriptors1 = sift.compute(gray1_sift, cv_keypoints1)
    cv_keypoints2, descriptors2 = sift.compute(gray2_sift, cv_keypoints2)

    if descriptors1 is None or descriptors2 is None:
        raise ValueError(f"SIFT descriptors could not be computed for {pair_name}.")
    if len(descriptors1) < 1 or len(descriptors2) < 2:
        raise ValueError(f"Not enough SIFT descriptors for {pair_name}.")

    print("SIFT descriptors:", descriptors1.shape, descriptors2.shape)

    # 5. Mean-center and normalize descriptors for NCC similarity.
    descriptors1 = descriptors1.astype(np.float32)
    descriptors2 = descriptors2.astype(np.float32)

    desc1_centered = descriptors1 - descriptors1.mean(axis=1, keepdims=True)
    desc2_centered = descriptors2 - descriptors2.mean(axis=1, keepdims=True)

    desc1_ncc = desc1_centered / (
        np.linalg.norm(desc1_centered, axis=1, keepdims=True) + 1e-10
    )
    desc2_ncc = desc2_centered / (
        np.linalg.norm(desc2_centered, axis=1, keepdims=True) + 1e-10
    )

    # 6. Best/second-best NCC in both directions, in batches.
    best_indices = np.empty(len(desc1_ncc), dtype=np.int32)
    best_values = np.empty(len(desc1_ncc), dtype=np.float32)
    second_values = np.empty(len(desc1_ncc), dtype=np.float32)
    reverse_best_indices = np.empty(len(desc2_ncc), dtype=np.int32)

    batch_size = 128

    for start in range(0, len(desc1_ncc), batch_size):
        stop = min(start + batch_size, len(desc1_ncc))
        scores = desc1_ncc[start:stop] @ desc2_ncc.T

        best_indices[start:stop] = np.argmax(scores, axis=1)
        top_two = np.partition(scores, kth=scores.shape[1] - 2, axis=1)[:, -2:]
        best_values[start:stop] = top_two.max(axis=1)
        second_values[start:stop] = top_two.min(axis=1)
        del scores

    for start in range(0, len(desc2_ncc), batch_size):
        stop = min(start + batch_size, len(desc2_ncc))
        scores = desc2_ncc[start:stop] @ desc1_ncc.T
        reverse_best_indices[start:stop] = np.argmax(scores, axis=1)
        del scores

    query_indices = np.arange(len(desc1_ncc))
    mutual_mask = reverse_best_indices[best_indices] == query_indices
    ncc_gap = best_values - second_values

    confidence_mask = (
        mutual_mask
        & (best_values >= min_ncc)
        & (ncc_gap >= min_ncc_gap)
    )

    print("Total descriptors:", len(desc1_ncc))
    print("Mutual matches:", int(np.count_nonzero(mutual_mask)))
    print("Confident mutual matches:", int(np.count_nonzero(confidence_mask)))

    selected = np.flatnonzero(confidence_mask)

    # If strict ambiguity filtering leaves too few matches, relax only the
    # NCC gap once; mutuality and minimum NCC similarity remain required.
    if len(selected) < 6 and min_ncc_gap > 0.005:
        relaxed_mask = mutual_mask & (best_values >= min_ncc) & (ncc_gap >= 0.005)
        selected = np.flatnonzero(relaxed_mask)
        print("Relaxed NCC-gap fallback matches:", len(selected))

    if len(selected) < 3:
        raise ValueError(
            f"Only {len(selected)} reliable matches remain for {pair_name}. "
            "The images may have too little common texture or the NCC threshold "
            "may need to be adjusted."
        )

    order = np.argsort(best_values[selected])[::-1]
    selected = selected[order[:top_n]]

    matches = np.column_stack((selected, best_indices[selected])).astype(np.int32)
    points1, points2 = get_matched_points(matches, cv_keypoints1, cv_keypoints2)
    print("Selected candidate matches:", len(matches))

    # 7. Custom affine RANSAC, with deterministic sampling.
    previous_rng_state = np.random.get_state()
    np.random.seed(random_seed)
    try:
        M, inliers = ransac_affine(
            points1,
            points2,
            iterations=ransac_iterations,
            threshold=ransac_threshold
        )
    finally:
        np.random.set_state(previous_rng_state)

    if M is None or inliers is None or np.count_nonzero(inliers) < 3:
        raise ValueError(f"RANSAC failed for {pair_name}.")

    # 8. Refine and evaluate.
    M_refined, final_inliers, mean_error, mse = refine_and_evaluate(
        points1, points2, M, inliers, threshold=ransac_threshold
    )

    valid_transform = (
        M_refined.shape == (2, 3)
        and np.isfinite(M_refined).all()
        and abs(np.linalg.det(M_refined[:, :2])) >= 1e-8
    )
    if not valid_transform:
        raise ValueError(f"Invalid/singular affine transformation for {pair_name}:\n{M_refined}")

    inlier_points1 = points1[final_inliers]
    inlier_points2 = points2[final_inliers]
    mapped_inlier_points2 = transform_points(inlier_points2, M_refined)
    inlier_errors = np.linalg.norm(inlier_points1 - mapped_inlier_points2, axis=1)

    median_inlier_error = float(np.median(inlier_errors))
    p95_inlier_error = float(np.percentile(inlier_errors, 95))

    spatial1 = summarize_inlier_spread(inlier_points1, image1.shape)
    spatial2 = summarize_inlier_spread(inlier_points2, image2.shape)

    inlier_count = int(np.count_nonzero(final_inliers))
    outlier_count = int(len(final_inliers) - inlier_count)
    inlier_ratio = inlier_count / max(len(matches), 1)

    print("Transform model: affine")
    print("Affine matrix:\n", M_refined)
    print("Inliers / outliers:", inlier_count, "/", outlier_count)
    print("Inlier ratio:", round(inlier_ratio, 4))
    print("Mean / median / P95 residual (px):",
          round(float(np.mean(inlier_errors)), 3),
          round(median_inlier_error, 3),
          round(p95_inlier_error, 3))
    print("Inlier grid cells (image 1 / image 2):",
          spatial1["occupied_cells"], "/", spatial2["occupied_cells"])
    print("Inlier hull coverage (image 1 / image 2):",
          f"{spatial1['hull_fraction']:.1%}", "/",
          f"{spatial2['hull_fraction']:.1%}")

    spatial_warnings = []
    for label, spread in (("image 1", spatial1), ("image 2", spatial2)):
        if spread["occupied_cells"] < 4 or spread["hull_fraction"] < 0.005:
            spatial_warnings.append(
                f"weak inlier spread in {label} "
                f"({spread['occupied_cells']}/16 cells; "
                f"{spread['hull_fraction']:.2%} hull)"
            )

    if spatial_warnings:
        print("ALIGNMENT WARNING:", "; ".join(spatial_warnings))

    if median_inlier_error > ransac_threshold or p95_inlier_error > 2 * ransac_threshold:
        raise ValueError(
            f"Unreliable affine fit for {pair_name}: median/P95 residual "
            f"{median_inlier_error:.2f}/{p95_inlier_error:.2f}px."
        )

    # 9. Generate panorama only after geometry has passed validation.
    panorama = create_blended_panorama(image1, image2, M_refined)

    return {
        "image1": image1,
        "image2": image2,
        "keypoints1": cv_keypoints1,
        "keypoints2": cv_keypoints2,
        "points1": points1,
        "points2": points2,
        "matches": matches,
        "inliers": final_inliers,
        "transformation": M_refined,
        "transform_model": "affine",
        "inlier_count": inlier_count,
        "outlier_count": outlier_count,
        "mean_error": float(np.mean(inlier_errors)),
        "mse": float(np.mean(inlier_errors ** 2)),
        "median_inlier_error": median_inlier_error,
        "p95_inlier_error": p95_inlier_error,
        "inlier_spatial": (spatial1, spatial2),
        "alignment_warnings": spatial_warnings,
        "panorama": panorama
    }


## Evaluate five image pairs

Pairs 1--4 cover the quantitative example and the three additional visual demonstrations. Pair 5 is included as an extra test.

### Final pipeline settings

The final run uses mutual-best NCC matches and rejects ambiguous matches before RANSAC. Ties favor lower median error, and the affine fit is refined iteratively. Blending is chunked; oversized canvases are rejected. The five input pairs must be in the \texttt{images/} folder.

In [ ]:
all_results = {}

for pair_name in image_pairs:
    results = process_image_pair(
        pair_name,
        top_n=300,
        ransac_iterations=2000,
        ransac_threshold=3.0,
        random_seed=42,
        min_ncc=0.75,
        min_ncc_gap=0.02
    )
    all_results[pair_name] = results

print("\nAll image pairs processed successfully!")

# Save full-resolution panoramas and alignment diagnostics.
for pair_name, results in all_results.items():
    safe_name = pair_name.lower().replace(" ", "_")

    panorama_path = OUTPUT_FOLDER / f"{safe_name}_panorama.png"
    if not cv2.imwrite(str(panorama_path), results["panorama"]):
        raise IOError(f"Could not save panorama to {panorama_path}")

    diagnostic_fig = plot_alignment_diagnostic(results, pair_name)
    diagnostic_path = OUTPUT_FOLDER / f"{safe_name}_alignment.png"
    diagnostic_fig.savefig(diagnostic_path, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(diagnostic_fig)

    print(f"Saved {panorama_path} and {diagnostic_path}")


### Compare the results

The figure summarizes the four Pair 1 descriptor tests and the inlier, error, and spatial-coverage metrics for all five pairs.

In [ ]:
# Place the Pair 1 method results and five-pair pipeline metrics in one figure.
method_rows = []
for method, (inliers, mean_error, mse) in method_comparison_results.items():
    method_rows.append([
        method, int(np.sum(inliers)), f"{float(mean_error):.2f}", f"{float(mse):.2f}"
    ])

pair_rows = []
for pair_name, pair_result in all_results.items():
    inlier_ratio = pair_result["inlier_count"] / max(len(pair_result["matches"]), 1)
    spread1, spread2 = pair_result["inlier_spatial"]
    pair_rows.append([
        pair_name, pair_result["transform_model"], pair_result["inlier_count"], f"{inlier_ratio:.2f}",
        f"{pair_result['median_inlier_error']:.2f}/{pair_result['p95_inlier_error']:.2f}",
        f"{spread1['hull_fraction']:.1%}/{spread2['hull_fraction']:.1%}",
        f"{spread1['occupied_cells']}/{spread2['occupied_cells']}"
    ])

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].axis("off")
axes[0].set_title("Pair 1 descriptor/matching comparison")
table1 = axes[0].table(
    cellText=method_rows,
    colLabels=["Method", "Inliers", "Mean err. (px)", "MSE (px²)"],
    loc="center", cellLoc="center"
)
table1.auto_set_font_size(False); table1.set_fontsize(9); table1.scale(1, 1.6)

axes[1].axis("off")
axes[1].set_title("Five-pair alignment and panorama comparison")
table2 = axes[1].table(
    cellText=pair_rows,
    colLabels=["Pair", "Model", "Inliers", "Ratio", "Median/P95 px", "Hull cov. I1/I2", "Grid cells I1/I2"],
    loc="center", cellLoc="center"
)
table2.auto_set_font_size(False); table2.set_fontsize(8); table2.scale(1, 1.6)
fig.tight_layout()
fig.savefig(OUTPUT_FOLDER / "comparison_tables.png", dpi=140, bbox_inches="tight")
plt.show()


### Save panorama and alignment figures

The notebook saves each panorama and its alignment diagnostic in \texttt{output/}, using filenames such as \texttt{pair\_1\_panorama.png} and \texttt{pair\_1\_alignment.png}.

In [ ]:
for pair_name, results in all_results.items():

    panorama_rgb = cv2.cvtColor(
        results["panorama"],
        cv2.COLOR_BGR2RGB
    )

    plt.figure(figsize=(16, 8))
    plt.imshow(panorama_rgb)
    plt.title(f"{pair_name} - Stitched Panorama")
    plt.axis("off")
    plt.show()